# 家計調査データ分析(Google Colab / Driveアップロード版)

**事前準備(このノートブックを開く前に1回だけ):**

1. ブラウザで https://drive.google.com を開く
2. マイドライブに `kakei_data` という名前の新規フォルダを作成
3. ローカルの `data/` フォルダにある以下の2ファイルを、そのフォルダにドラッグ&ドロップでアップロード
   - `kakei_savings_debt_by_income.csv`
   - `kakei_surplus_rate_by_income_quintile.csv`

アップロードが終わったら、下のセルを上から順に実行してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/kakei_data"

import os
print(os.listdir(DATA_DIR))  # アップロードした2つのCSVが表示されればOK

In [ ]:
import pandas as pd

df_savings = pd.read_csv(f"{DATA_DIR}/kakei_savings_debt_by_income.csv")
df_surplus = pd.read_csv(f"{DATA_DIR}/kakei_surplus_rate_by_income_quintile.csv")

print("貯蓄・負債データ:", df_savings.shape)
display(df_savings.head())

print("\n黒字率データ:", df_surplus.shape)
display(df_surplus.head())

## 年間収入階級別の貯蓄推移を見る

In [ ]:
import matplotlib.pyplot as plt

pivot_savings = df_savings[df_savings["項目"] == "貯蓄"].pivot_table(
    index="時期", columns="年間収入階級", values="値"
)
pivot_savings.plot(figsize=(11, 5), title="年間収入階級別 貯蓄現在高の推移(万円)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 年収五分位別の黒字率(貯蓄できている度合い)を比較する

In [ ]:
df_rate = df_surplus[
    (df_surplus["項目"] == "黒字率") & (df_surplus["年間収入五分位"] != "平均")
].copy()

df_rate["年"] = df_rate["時期"].astype(str).str[:4]
yearly_avg = df_rate.groupby(["年", "年間収入五分位"])["値"].mean().unstack()

yearly_avg.plot(figsize=(11, 5), title="年収五分位別 平均黒字率の推移(%)")
plt.ylabel("黒字率(%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("\n直近年の年収五分位別 平均黒字率:")
print(yearly_avg.tail(1).T)

## 平均消費性向 vs 黒字率(貯蓄しやすい世帯の特徴を探る)

In [ ]:
df_compare = df_surplus[
    df_surplus["項目"].isin(["平均消費性向", "黒字率"])
    & (df_surplus["年間収入五分位"] != "平均")
].copy()
df_compare["年"] = df_compare["時期"].astype(str).str[:4]

summary = df_compare.groupby(["年間収入五分位", "項目"])["値"].mean().unstack()
summary = summary.reindex(["年収五分位1", "年収五分位2", "年収五分位3", "年収五分位4", "年収五分位5"])
print(summary)

summary.plot(kind="bar", figsize=(9, 5), title="年収五分位別 平均消費性向・黒字率(全期間平均)")
plt.ylabel("%")
plt.tight_layout()
plt.show()